# Løhre & Jørgensen 2014 — Numerical Anchors and Their Strong Effects on Software Development Effort Estimates

In [5]:
import re, utils

DOCUMENTS = utils.load("project brief")

def build_lohre_tasks(conditions):
    """Return (tasks, result_store) ready for fire_and_collect."""
    tasks = []
    for doc_name, doc_text in DOCUMENTS:
        for condition, anchor_sentence in conditions.items():
            prompt = f"""
You are a software developer asked to estimate the effort required to complete the
project below. Assume you will carry out the development work yourself, using the
programming language, tools, and database you know best.
{anchor_sentence}

```requirements
{doc_text}
```

End your response with exactly:
MOST_LIKELY: <number> work-hours
MINIMUM: <number> work-hours
MAXIMUM: <number> work-hours
"""
            for model in utils.MODELS:
                tasks.append((model, prompt, {"doc": doc_name, "condition": condition}))
    return tasks

def merge_lohre(store, model, response, meta):
    doc, condition = meta["doc"], meta["condition"]
    ml = re.search(r"MOST_LIKELY:\s*(\d+)", response or "")
    mn = re.search(r"MINIMUM:\s*(\d+)",     response or "")
    mx = re.search(r"MAXIMUM:\s*(\d+)",     response or "")
    ml, mn, mx = (int(m.group(1)) if m else None for m in (ml, mn, mx))
    store.setdefault(model, {}).setdefault(doc, {})[condition] = {
        "most_likely": ml, "minimum": mn, "maximum": mx,
        "pi_width": round((mx - mn) / ml, 3) if (ml and mn and mx and ml > 0) else None,
    }
    print(f"  {model.split('/')[1]:20} | {doc:30} | {condition}")


## Experiment 1 — High anchor, anchor precision

Five groups. Anchor values cluster around 1000 hours (implausibly high). Precision varies: precise single (998 h), round single (1000 h), precise interval (900–1100 h), imprecise interval (500–1500 h).

In [ ]:
exp1_conditions = {
    "control":            "",
    "precise_single":     "\nBefore you begin: consider how likely it is that this project will take less than 998 work-hours.",
    "round_single":       "\nBefore you begin: consider how likely it is that this project will take less than 1000 work-hours.",
    "precise_interval":   "\nBefore you begin: consider how likely it is that this project will take between 900 and 1100 work-hours.",
    "imprecise_interval": "\nBefore you begin: consider how likely it is that this project will take between 500 and 1500 work-hours.",
}
results_exp1 = {}
utils.fire_and_collect(build_lohre_tasks(exp1_conditions), results_exp1, merge_lohre,
                       save_path="lohre2014_exp1_partial.json")


## Experiment 2 — Low anchor, anchor precision

Three groups. Anchor values are low (~20 hours). Precise interval (19–21 h) vs imprecise interval (10–30 h).

In [ ]:
exp2_conditions = {
    "control":            "",
    "precise_interval":   "\nBefore you begin: consider how likely it is that this project will take between 19 and 21 work-hours.",
    "imprecise_interval": "\nBefore you begin: consider how likely it is that this project will take between 10 and 30 work-hours.",
}
results_exp2 = {}
utils.fire_and_collect(build_lohre_tasks(exp2_conditions), results_exp2, merge_lohre,
                       save_path="lohre2014_exp2_partial.json")


## Experiment 3 — Source credibility

Four groups. Anchor is always 10 hours. Source varies: neutral framing, low-credibility source (admin with no technical background), high-credibility source (project manager with software background).

In [ ]:
exp3_conditions = {
    "control":          "",
    "low_credibility":  "\nAn administrative person in your company, with no background in software development, is responsible for logging all work over 10 hours. Without looking at the requirements, they ask whether you think this will take less than 10 work-hours.",
    "high_credibility": "\nYour project manager, who has a background in software development, has reviewed the requirements and asks whether you think this will take more than 10 work-hours.",
    "neutral":          "\nDo you think this project will take more than 10 work-hours?",
}
results_exp3 = {}
utils.fire_and_collect(build_lohre_tasks(exp3_conditions), results_exp3, merge_lohre,
                       save_path="lohre2014_exp3_partial.json")

utils.save("lohre2014_results.json", {"exp1": results_exp1, "exp2": results_exp2, "exp3": results_exp3})
